# English 5-Letter Word Ladder — Distance Regression Training Data

Generate training examples for fine-tuning BERT to predict **BFS shortest-path distance** between word pairs in the word-ladder graph.

- **Full vocab**: `data/islands/english_5_largest_island.txt` (largest connected component)
- **Format**: regression — input `(word_a, word_b)`, label = shortest-path distance (integer)
- **Output**: `data/training/wordladder_english5_{train,val,test}.csv`
- **Split**: by word pair; stratified by distance bin

### Why distance regression?
The previous binary-classification approach (is candidate the correct next step?) reached 82% accuracy but failed at path generation — compounding per-step errors, poor ranking calibration, and no graph awareness. Distance regression directly predicts what A\* search needs: *how far is this word from the target?* At inference we score every neighbor and pick the one with the lowest predicted distance.

In [2]:
import random
from pathlib import Path

import networkx as nx
import pandas as pd

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

SEED = 42
random.seed(SEED)

BASE = Path("../data")
ISLANDS = BASE / "islands"
TRAINING = BASE / "training"
TRAINING.mkdir(parents=True, exist_ok=True)

FULL_VOCAB = ISLANDS / "english_5_largest_island.txt"


def load_words(path: Path) -> set[str]:
    return {w.strip().lower() for w in path.read_text(encoding="utf-8").splitlines() if w.strip()}


def one_letter_neighbors(w: str, vocab: set[str]) -> set[str]:
    import string
    neighbors = set()
    for i in range(len(w)):
        for c in string.ascii_lowercase:
            if c != w[i]:
                cand = w[:i] + c + w[i + 1:]
                if cand in vocab:
                    neighbors.add(cand)
    return neighbors


vocab = load_words(FULL_VOCAB)
print(f"Full vocab: {FULL_VOCAB.name} ({len(vocab)} words)")

Playable: english_5_strict_largest_island.txt (5330 words)
Full vocab: english_5_largest_island.txt (9902 words)


In [3]:
# 2. Build graph from full vocab
G = nx.Graph()
G.add_nodes_from(vocab)
for w in tqdm(vocab, desc="Building graph"):
    for nb in one_letter_neighbors(w, vocab):
        if w < nb:
            G.add_edge(w, nb)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

Building graph: 100%|██████████| 9902/9902 [00:01<00:00, 9619.97it/s] 

Nodes: 9902, Edges: 34204


In [4]:
# 3. Compute BFS distances from sampled source words
# Running single-source BFS from N words gives N × |V| distances efficiently.
NUM_SOURCES = 2000
vocab_list = sorted(vocab)
sources = random.sample(vocab_list, NUM_SOURCES)

raw_pairs = []
for src in tqdm(sources, desc="BFS from source words"):
    lengths = nx.single_source_shortest_path_length(G, src)
    for tgt, dist in lengths.items():
        if tgt != src:
            raw_pairs.append((src, tgt, dist))

# Deduplicate symmetric pairs: keep only one of (a,b) and (b,a)
seen = set()
unique_pairs = []
for a, b, d in raw_pairs:
    key = (min(a, b), max(a, b))
    if key not in seen:
        seen.add(key)
        unique_pairs.append((a, b, d))
raw_pairs = unique_pairs

print(f"Unique pairs from {NUM_SOURCES} BFS sources: {len(raw_pairs)}")

Sampled 18000 unique (start, target) pairs


In [5]:
# 4. Subsample with balanced distance coverage
from collections import Counter, defaultdict

TARGET_EXAMPLES = 250_000

by_dist = defaultdict(list)
for item in raw_pairs:
    by_dist[item[2]].append(item)

dist_values = sorted(by_dist.keys())
per_dist = TARGET_EXAMPLES // len(dist_values)

sampled = []
for d in dist_values:
    pool = by_dist[d]
    n = min(len(pool), per_dist)
    sampled.extend(random.sample(pool, n))

if len(sampled) < TARGET_EXAMPLES:
    used = {(a, b) for a, b, _ in sampled}
    extra = [(a, b, d) for a, b, d in raw_pairs if (a, b) not in used]
    random.shuffle(extra)
    sampled.extend(extra[: TARGET_EXAMPLES - len(sampled)])

random.shuffle(sampled)
sampled = sampled[:TARGET_EXAMPLES]

# Randomly swap word order (distance is symmetric; model should learn both directions)
examples = []
for a, b, d in sampled:
    if random.random() < 0.5:
        a, b = b, a
    examples.append({"text_a": a, "text_b": b, "label": d})

print(f"Final examples: {len(examples)}")

dist_counts = Counter(ex["label"] for ex in examples)
print("\nDistance distribution:")
for d in sorted(dist_counts):
    print(f"  d={d}: {dist_counts[d]:>6} ({dist_counts[d] / len(examples):.1%})")

Finding paths: 100%|██████████| 18000/18000 [00:10<00:00, 1761.14it/s]

Valid paths (3–10 steps): 15928


In [ ]:
# 5. Split by pair — stratified by distance bin (short 1–3, medium 4–6, long 7+)

pair_to_dist = {(ex["text_a"], ex["text_b"]): ex["label"] for ex in examples}
pairs_list = list(pair_to_dist.keys())
dist_bins = [0 if pair_to_dist[p] <= 3 else (1 if pair_to_dist[p] <= 6 else 2) for p in pairs_list]

bins_to_pairs = defaultdict(list)
for p, b in zip(pairs_list, dist_bins):
    bins_to_pairs[b].append(p)

train_set, val_set, test_set = set(), set(), set()
for bin_id, bin_pairs in bins_to_pairs.items():
    random.shuffle(bin_pairs)
    n = len(bin_pairs)
    t1 = int(0.90 * n)
    t2 = int(0.95 * n)
    train_set.update(bin_pairs[:t1])
    val_set.update(bin_pairs[t1:t2])
    test_set.update(bin_pairs[t2:])

train_data = [ex for ex in examples if (ex["text_a"], ex["text_b"]) in train_set]
val_data = [ex for ex in examples if (ex["text_a"], ex["text_b"]) in val_set]
test_data = [ex for ex in examples if (ex["text_a"], ex["text_b"]) in test_set]

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    pd.DataFrame(data).to_csv(TRAINING / f"wordladder_english5_{name}.csv", index=False)

print(f"\n=== Final stats ===")
print(f"Total: {len(examples)}")
print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")
print(f"Label range: {min(ex['label'] for ex in examples)}–{max(ex['label'] for ex in examples)}")
print(f"Mean distance: {sum(ex['label'] for ex in examples) / len(examples):.2f}")
print(f"Saved to {TRAINING}/")

Creating examples: 100%|██████████| 15928/15928 [01:07<00:00, 235.15it/s]


After dedup: 406806 examples
Capped to 15000 examples


In [7]:
# Done — CSVs ready for notebook 06 (BERT distance regression fine-tuning).
print("Data generation complete. Run notebook 06 to fine-tune.")


=== Final stats ===
Valid pairs: 15928
Unique (start,target) in data: 9479
Total examples: 15000
Train: 13444 (pos: 3385, neg: 10059)
Val: 785 (pos: 206)
Test: 771 (pos: 173)
Class balance (train): 25.18% pos, 74.82% neg
Saved to ..\data\training/ (CSV)
